In [25]:
import os, sys
project_root = os.path.dirname(os.getcwd())
# print(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

from src.data_ingestion import load_processed_data_from_gcs

sns.set_theme(style="whitegrid")
PALETTE = {"positive": "#2ecc71", "neutral": "#ea890b", "negative": "#e90e07"}

os.makedirs("artifacts", exist_ok=True)



In [3]:
# load the data which is already preprocessed.
df = load_processed_data_from_gcs("processed_reviews.csv")
df = df.dropna(subset=["cleaned_review"]).reset_index(drop=True)

print(df.shape)


(17321, 4)


In [4]:
df['sentiments'].value_counts()

sentiments
positive    9503
neutral     6284
negative    1534
Name: count, dtype: int64

In [5]:
le = LabelEncoder()
df["label"] = le.fit_transform(df['sentiments'])
print("Label encoding..")
print(dict(zip(le.classes_, le.transform(le.classes_))))

Label encoding..
{'negative': np.int64(0), 'neutral': np.int64(1), 'positive': np.int64(2)}


In [6]:
df

,sentiments,cleaned_review,review_score,word_count,label
0,positive,i wish would have gotten one earlier love it a...,5.0,19,2
1,neutral,i ve learned this lesson again open the packag...,1.0,88,1
2,neutral,it is so slow and lags find better option,2.0,9,1
3,neutral,roller ball stopped working within months of m...,1.0,12,1
4,neutral,i like the color and size but it few days out ...,1.0,21,1
...,...,...,...,...,...
17316,positive,i love this speaker and love can take it anywh...,5.0,30,2
17317,positive,i use it in my house easy to connect and loud ...,4.0,13,2
17318,positive,the bass is good and the battery is amazing mu...,5.0,41,2
17319,positive,love it,5.0,2,2


In [7]:
X_text = df['cleaned_review']
X_num = df[["review_score", "word_count"]]
y = df["label"]

In [8]:
X_tr_text, X_te_text, X_tr_num, X_te_num, y_train, y_test = train_test_split(X_text, X_num, y, test_size=0.2, random_state=42, stratify=y)

In [9]:
X_tr_text.shape, X_tr_num.shape, y_train.shape, y_test.shape, X_te_text.shape, X_te_num.shape

((13856,), (13856, 2), (13856,), (3465,), (3465,), (3465, 2))

In [10]:
# Vectorization  ----> Count Vectorizer -----> also called BAG OF WORDS
# documents = ["it felt like it would break easily and the texture of the mouse would get dirty quickly", "I am not very happy with the product"]
# {"it": 2, "felt": 1, "would": 2, "easily": 1, "and": 1, "the": 2, "texture": 1, "of": 1, "mouse": 1, "get": 1, "dirty": 1, "quickly": 1}
# cv = CountVectorizer()
# feature_matrix = cv.fit_transform(documents)
# print(feature_matrix.toarray())

cv = CountVectorizer(max_features=10000, ngram_range=(1, 2))
X_tr_cv = cv.fit_transform(X_tr_text)
X_te_cv = cv.transform(X_te_text)











In [11]:
len(cv.vocabulary_.keys())

10000

In [12]:
X_te_cv

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 134099 stored elements and shape (3465, 10000)>

In [13]:
X_te_cv.toarray()

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(3465, 10000))

In [14]:
# TF-IDF --> Term frequency inverse document frequency

# 1. TF ---> Term frequency
# TF = Number of times the word/term appear in the document/total number of terms/words in the document

# [i have had this few months and loved it2 but now the roller to scroll is not working. so I have had enough of it and don't want to get it again. ]

# total_words_in_document = 18

# TF_have = 2/26 = 0.076
# TF_had = 2/26 = 0.076
# TF_this = 1/26 = 0.038
# TF_few = 1/26 = 0.038
# TF_it = 3/26


# IDF = Inverse document frequency 
# IDF = log(N/DF)
# N = Total number of documents = 2
# DF = The number of documents that contains the word

# IDF(have) = log(2/2) = 0
# IDF(this) = log(2/1) = 0.6
# IDF(few) = log(2/1) = 0.6
# IDF(it) = log(2/2) = 0


# TF_IDF = TF * IDF

# TFIDF(have) = 0.076 * 0 = 0
# TFIDF(had) = 0
# TFIDF(this) = 0.038 * 0.6 = 0.022
# TFIDF(few) = 0.038 * 0.6 = 0.022




In [15]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X_tr_tfidf = tfidf.fit_transform(X_tr_text)
X_te_tfidf = tfidf.transform(X_te_text)


In [16]:
tfidf.vocabulary_

{'bad': np.int64(830),
 'ass': np.int64(732),
 'headset': np.int64(3409),
 'love': np.int64(4821),
 'it': np.int64(3951),
 'love it': np.int64(4825),
 'bought': np.int64(1117),
 'this': np.int64(8356),
 'mouse': np.int64(5136),
 'for': np.int64(2676),
 'my': np.int64(5294),
 'work': np.int64(9705),
 'laptop': np.int64(4476),
 'the': np.int64(7864),
 'right': np.int64(6897),
 'click': np.int64(1612),
 'on': np.int64(5836),
 'is': np.int64(3772),
 'constantly': np.int64(1800),
 'getting': np.int64(2975),
 'stuck': np.int64(7604),
 'and': np.int64(289),
 'left': np.int64(4562),
 'seems': np.int64(7032),
 'to': np.int64(8523),
 'take': np.int64(7695),
 'several': np.int64(7084),
 'before': np.int64(984),
 'what': np.int64(9399),
 'supposed': np.int64(7653),
 'do': np.int64(2120),
 'overall': np.int64(6149),
 'wouldn': np.int64(9875),
 'buy': np.int64(1294),
 'again': np.int64(147),
 'would': np.int64(9841),
 'definitely': np.int64(1999),
 'recommend': np.int64(6758),
 'spending': np.int64(

In [17]:
X_te_tfidf.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(3465, 10000))

In [18]:
X_te_tfidf.toarray()[X_te_tfidf.toarray() != 0]

array([0.37893743, 0.65523605, 0.19306975, ..., 0.20071957, 0.10510415,
       0.25086481], shape=(134099,))

In [19]:
y_test

17049    2
5187     1
11922    2
1898     2
15755    2
        ..
13180    2
7699     1
17317    2
4423     2
1884     1
Name: label, Length: 3465, dtype: int64

In [20]:
X_train = hstack([X_tr_tfidf, csr_matrix(X_tr_num)])
X_test = hstack([X_te_tfidf, csr_matrix(X_te_num)])

print("FInal feature matrix")

print(X_train.shape)
print(X_test.shape)

FInal feature matrix
(13856, 10002)
(3465, 10002)


In [21]:
X_train.toarray()

array([[ 0.,  0.,  0., ...,  0.,  5.,  5.],
       [ 0.,  0.,  0., ...,  0.,  5., 56.],
       [ 0.,  0.,  0., ...,  0.,  1., 48.],
       ...,
       [ 0.,  0.,  0., ...,  0.,  5., 28.],
       [ 0.,  0.,  0., ...,  0.,  5.,  4.],
       [ 0.,  0.,  0., ...,  0.,  1.,  4.]], shape=(13856, 10002))

In [22]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 584366 stored elements and shape (13856, 10002)>

In [23]:
y_train

13126    1
1117     2
7629     1
7932     2
11219    1
        ..
16535    2
5380     2
8662     1
13611    1
10765    2
Name: label, Length: 13856, dtype: int64

In [26]:
import joblib

joblib.dump(X_tr_tfidf, "artifacts/X_train.pkl")
joblib.dump(X_te_tfidf, "artifacts/X_test.pkl")

joblib.dump(y_train, "artifacts/y_train.pkl")
joblib.dump(y_test, "artifacts/y_test.pkl")


['artifacts/y_test.pkl']

In [31]:
# tfidf
joblib.dump(tfidf, "../models/tfidf_vectorizer.pkl") 
joblib.dump(le, "../models/label_encoder.pkl")

['../models/label_encoder.pkl']

In [77]:
le

Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)Holds the label for each class.","ndarray[object](3,)","['negative','neutral','positive']"


In [78]:
from pathlib import Path
model_dir = Path("../models")

for file in model_dir.glob("*.pkl"):
    print(file.name)

label_encoder.pkl
linear_svc.pkl
logistic_regression.pkl
random_forest.pkl
tfidf_vectorizer.pkl
